# Qwen3-TTS — TRUE Live Streaming in Colab (vLLM-Omni)

This notebook runs Qwen3-TTS behind **vLLM-Omni** and tests:
1. OpenAI-compatible `/v1/audio/speech`
2. real chunked PCM streaming
3. time-to-first-audio-byte
4. a browser Gradio streaming demo

**Use a fresh Colab runtime for this notebook.** It installs vLLM-Omni and should not share a runtime with the direct `qwen-tts` notebook.


In [ ]:
# 1) GPU
!nvidia-smi


In [ ]:
# 2) Install vLLM-Omni + demo dependencies
!pip -q install -U "vllm-omni[demo]" requests soundfile websockets
!rm -rf /content/vllm-omni-src
!git clone -q --depth 1 https://github.com/vllm-project/vllm-omni.git /content/vllm-omni-src


In [ ]:
# 3) Start Qwen3-TTS server
import os, subprocess, time, requests
MODEL = "Qwen/Qwen3-TTS-12Hz-1.7B-CustomVoice"
PORT = 8091
LOG = "/content/qwen_vllm.log"
try:
    server.terminate()
except Exception:
    pass
logf = open(LOG, "w")
server = subprocess.Popen(["vllm", "serve", MODEL, "--omni", "--port", str(PORT)], stdout=logf, stderr=subprocess.STDOUT)
print("Starting server...")
ready = False
for i in range(180):
    time.sleep(2)
    try:
        r = requests.get(f"http://127.0.0.1:{PORT}/v1/models", timeout=2)
        if r.ok:
            ready = True
            break
    except Exception:
        pass
    if server.poll() is not None:
        break
if not ready:
    print(open(LOG).read()[-8000:])
    raise RuntimeError("Server did not become ready. Check the log above.")
print("Server ready:", f"http://127.0.0.1:{PORT}")


In [ ]:
# 4) Normal API request
import requests, time
from IPython.display import Audio, display
payload = {
    "model": MODEL,
    "input": "こんにちは！今日は一緒に自然な日本語の会話を練習しましょう。",
    "voice": "Ono_Anna",
    "language": "Japanese",
    "instructions": "親しみやすく自然な会話調。",
    "response_format": "wav",
}
t0 = time.perf_counter()
r = requests.post(f"http://127.0.0.1:{PORT}/v1/audio/speech", json=payload, timeout=300)
elapsed = time.perf_counter() - t0
r.raise_for_status()
OUT = "/content/qwen_api.wav"
open(OUT, "wb").write(r.content)
print(f"HTTP generation completed in {elapsed:.3f}s; bytes={len(r.content):,}")
display(Audio(OUT))


In [ ]:
# 5) TRUE chunked PCM streaming + TTFB measurement
import requests, time, wave
payload = {
    "model": MODEL,
    "input": "えっ、本当ですか？それはすごいですね。もう少し詳しく教えてください。",
    "voice": "Ono_Anna",
    "language": "Japanese",
    "instructions": "驚きのある自然な会話調。",
    "stream": True,
    "stream_format": "audio",
    "response_format": "pcm",
}
t0 = time.perf_counter()
first_byte_at = None
chunks = []
with requests.post(f"http://127.0.0.1:{PORT}/v1/audio/speech", json=payload, stream=True, timeout=300) as r:
    r.raise_for_status()
    for chunk in r.iter_content(chunk_size=4096):
        if not chunk:
            continue
        if first_byte_at is None:
            first_byte_at = time.perf_counter()
        chunks.append(chunk)
done = time.perf_counter()
pcm = b"".join(chunks)
OUT = "/content/qwen_stream.wav"
with wave.open(OUT, "wb") as wf:
    wf.setnchannels(1)
    wf.setsampwidth(2)
    wf.setframerate(24000)
    wf.writeframes(pcm)
print(f"TTFB              : {first_byte_at - t0:.3f}s")
print(f"Total request time: {done - t0:.3f}s")
print(f"PCM bytes         : {len(pcm):,}")
display(Audio(OUT))


In [ ]:
# 6) Launch official browser streaming demo
import subprocess, os, time
DEMO = "/content/vllm-omni-src/examples/online_serving/text_to_speech/qwen3_tts/gradio_demo.py"
demo_log = open("/content/qwen_gradio.log", "w")
try:
    demo_proc.terminate()
except Exception:
    pass
demo_proc = subprocess.Popen(["python", DEMO, "--api-base", f"http://127.0.0.1:{PORT}", "--host", "0.0.0.0", "--port", "7860", "--share"], cwd=os.path.dirname(DEMO), stdout=demo_log, stderr=subprocess.STDOUT)
time.sleep(8)
print(open("/content/qwen_gradio.log").read()[-4000:])
print("If the public URL has not appeared yet, rerun: print(open('/content/qwen_gradio.log').read()[-4000:])")


In [ ]:
# 7) Logs / cleanup
print(open("/content/qwen_vllm.log").read()[-5000:])
# server.terminate()
# demo_proc.terminate()
